In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# -----------------------------
# 1. Load tokenizer & model
# -----------------------------
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# -----------------------------
# 2. Prepare dataset
# -----------------------------
texts = [
    "I love this product",
    "This is terrible",
    "Amazing experience",
    "Worst service ever"
]

labels = [1, 0, 1, 0]  # 1 = positive, 0 = negative

dataset = Dataset.from_dict({
    "text": texts,
    "label": labels
})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# -----------------------------
# 3. Training configuration
# -----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
)

# -----------------------------
# 4. Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# -----------------------------
# 5. Train
# -----------------------------
trainer.train()

# -----------------------------
# 6. Save model
# -----------------------------
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")
